In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [1]:
import spacy

# 加载英文模型
en_nlp = spacy.load("en_core_web_sm")
print("✓ English model loaded")

# 加载德文模型
de_nlp = spacy.load("de_core_news_sm")
print("✓ German model loaded")

# 测试英文
en_doc = en_nlp("Hello world!")
print(f"English: {[token.text for token in en_doc]}")

# 测试德文
de_doc = de_nlp("Hallo Welt!")
print(f"German: {[token.text for token in de_doc]}")

# 更复杂的测试
text = "Apple is looking to buy a startup in London for $1 million"
doc = en_nlp(text)
print("\nNamed Entities:")
for ent in doc.ents:
    print(f"  {ent.text} -> {ent.label_}")

✓ English model loaded
✓ German model loaded
English: ['Hello', 'world', '!']
German: ['Hallo', 'Welt', '!']

Named Entities:
  Apple -> ORG
  London -> GPE
  $1 million -> MONEY


# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

In [102]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

1.答：改变了 Hugging Face 库连接的服务器地址，将默认的官方 Hub 端点从 https://huggingface.co 切换到国内的镜像站点。

2.答：因为用镜像更加稳定，下载速度也更快

3.答：这行代码加载了Multi30k 数据集

## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [25]:
import datasets  # 必须导入

# 加载数据集
dataset = datasets.load_dataset("bentrevett/multi30k")

# 提取数据
train_data = dataset["train"]
valid_data = dataset["validation"]
test_data = dataset["test"]

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(valid_data)} 条")
print(f"测试集: {len(test_data)} 条")

训练集: 29000 条
验证集: 1014 条
测试集: 1000 条


In [26]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [27]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [2]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [3]:
import spacy

# 先加载模型
en_nlp = spacy.load("en_core_web_sm")

# 然后再使用
string = "What a lovely day it is today!"
tokens = [token.text for token in en_nlp.tokenizer(string)]
print(tokens)

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']


## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


In [30]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

作用是将原始文本数据转换为分词后的标记序列，为神经机器翻译（NMT）模型准备输入数据。

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


In [6]:
import spacy
from datasets import Dataset
import pandas as pd

# 1. 加载 spaCy 模型
print("Loading spaCy models...")
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")
print("✓ Models loaded")

# 2. 创建足够大的数据集（至少10个样本）
sample_data = {
    'en': [
        "What a lovely day it is today!",
        "Hello world, how are you?",
        "The weather is nice today.",
        "I love programming in Python",
        "This is a test sentence.",
        "Good morning everyone!",
        "Have a nice day!",
        "See you tomorrow.",
        "Thank you very much.",
        "You are welcome."
    ],
    'de': [
        "Was für ein schöner Tag heute ist!",
        "Hallo Welt, wie geht es dir?",
        "Das Wetter ist schön heute.",
        "Ich liebe es, in Python zu programmieren",
        "Das ist ein Testsatz.",
        "Guten Morgen zusammen!",
        "Hab einen schönen Tag!",
        "Bis morgen.",
        "Vielen Dank.",
        "Bitte schön."
    ]
}

df = pd.DataFrame(sample_data)
dataset = Dataset.from_pandas(df)
print(f"✓ 数据集大小: {len(dataset)}")

# 3. 分割数据集（现在有10个样本，可以正常分割）
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_data = split_dataset["train"]
temp_data = split_dataset["test"]
valid_test = temp_data.train_test_split(test_size=0.5, seed=42)
valid_data = valid_test["train"]
test_data = valid_test["test"]

print(f"训练集: {len(train_data)}")
print(f"验证集: {len(valid_data)}")
print(f"测试集: {len(test_data)}")

# 4. 定义 tokenize 函数
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_text = example['en']
    de_text = example['de']
    
    if lower:
        en_text = en_text.lower()
        de_text = de_text.lower()
    
    en_tokens = [token.text for token in en_nlp.tokenizer(en_text)]
    de_tokens = [token.text for token in de_nlp.tokenizer(de_text)]
    
    # 截断并添加特殊token
    en_tokens = [sos_token] + en_tokens[:max_length-2] + [eos_token]
    de_tokens = [sos_token] + de_tokens[:max_length-2] + [eos_token]
    
    return {
        'en_tokens': en_tokens,
        'de_tokens': de_tokens,
        'en_length': len(en_tokens),
        'de_length': len(de_tokens)
    }

# 5. 参数设置
max_length = 100
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# 6. 应用 tokenization
print("\nTokenizing training data...")
train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)

print("Tokenizing validation data...")
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)

print("Tokenizing test data...")
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

print("\n✓ Tokenization complete!")

# 7. 显示示例
print("\n示例:")
print(f"原文: {train_data[0]['en']}")
print(f"Tokens: {train_data[0]['en_tokens'][:10]}...")
print(f"德文 Tokens: {train_data[0]['de_tokens'][:10]}...")

Loading spaCy models...
✓ Models loaded
✓ 数据集大小: 10
训练集: 8
验证集: 1
测试集: 1

Tokenizing training data...


Map: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 428.14 examples/s]


Tokenizing validation data...


Map: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 61.63 examples/s]


Tokenizing test data...


Map: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 70.09 examples/s]


✓ Tokenization complete!

示例:
原文: What a lovely day it is today!
Tokens: ['<sos>', 'what', 'a', 'lovely', 'day', 'it', 'is', 'today', '!', '<eos>']...
德文 Tokens: ['<sos>', 'was', 'für', 'ein', 'schöner', 'tag', 'heute', 'ist', '!', '<eos>']...


sos的含义是序列开始标记 eos的含义是序列结束标记 map函数的作用是对数据集中的每个样本应用指定的处理函数

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [110]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [111]:
import spacy
import datasets

# 1. 加载 spaCy 模型
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

# 2. 定义分词函数
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

# 3. 设置参数
max_length = 1000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# 4. 重新加载原始数据（如果已被污染）
dataset = datasets.load_dataset("bentrevett/multi30k")
train_data = dataset["train"]
valid_data = dataset["validation"]
test_data = dataset["test"]

# 5. 执行分词
train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

# 6. 验证
print(train_data.column_names)  # 应该包含 'en_tokens' 和 'de_tokens'
print(train_data[0])  # 查看第一条样本

['en', 'de', 'en_tokens', 'de_tokens']
{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.', 'en_tokens': ['<sos>', 'two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<eos>'], 'de_tokens': ['<sos>', 'zwei', 'junge', 'weiße', 'männer', 'sind', 'im', 'freien', 'in', 'der', 'nähe', 'vieler', 'büsche', '.', '<eos>']}


In [112]:
import torchtext
from torchtext.vocab import build_vocab_from_iterator

# 所有变量必须先定义
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
sos_token = "<sos>"  # ⭐ 必须定义
eos_token = "<eos>"  # ⭐ 必须定义

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

# 设置默认索引（重要！）
en_vocab.set_default_index(en_vocab[unk_token])
de_vocab.set_default_index(de_vocab[unk_token])

print(f"英文词汇表: {len(en_vocab)} 词")
print(f"德文词汇表: {len(de_vocab)} 词")

英文词汇表: 5893 词
德文词汇表: 7853 词


## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [113]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [114]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [115]:
en_vocab["the"]

7

In [116]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [117]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [118]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [119]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

In [ ]:
如果 "crime" 在训练集中只出现 1 次，它会被排除在词汇表外，词汇表只包含高频词，低频词被替换为 <unk>

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


In [120]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [121]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [122]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [123]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [31]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

In [47]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [ ]:
get_collate_fn 函数的核心作用

解决变长序列问题，通过填充使批次内所有序列长度一致。

提高计算效率，批量处理而非逐个处理样本。

保持数据完整性，保留原始的索引信息，只添加填充值。

灵活配置，通过闭包机制传递填充索引参数。

In [48]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [ ]:
用于批量加载序列数据，并自动处理变长序列的填充问题。

In [49]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

In [ ]:
批量大小，训练数据加载，

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

In [35]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

In [ ]:
Encoder 的核心任务是将变长的输入序列压缩为固定长度的上下文向量，
这些向量捕捉了整个输入序列的语义信息，为解码器的生成提供条件。
通过多层 LSTM 和 Dropout 机制，编码器能够有效处理序列数据并防止过拟合。

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

In [36]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

In [ ]:
编码器提供的上下文（初始隐藏状态）

根据已生成的词预测下一个词

输出目标词汇表上的概率分布

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

In [37]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

In [ ]:
编码器-解码器架构用于编码器理解输入，解码器生成输出

循环生成使解码器逐个时间步生成输出词

Teacher Forcing令平衡训练速度和模型质量的重要技术
outputs 张量保存所有时间步的预测结果，用于计算损失

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [11]:
import torch
import torch.nn as nn

# 编码器初始化
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

# 解码器初始化
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

# Seq2Seq 模型
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        _, hidden, cell = self.encoder(src)
        
        input = trg[:, 0]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
            
        return outputs

In [12]:
import torch
import torch.nn as nn
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\torch\cuda\__init__.py:209: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [8]:
from collections import Counter

def build_vocab(data, token_field, min_freq=1, max_size=None, special_tokens=None):
    """构建词汇表"""
    if special_tokens is None:
        special_tokens = ['<pad>', '<unk>', '<sos>', '<eos>']
    
    # 统计词频
    counter = Counter()
    for example in data:
        tokens = example[token_field]
        counter.update(tokens)
    
    # 创建词汇表
    vocab = {token: idx for idx, token in enumerate(special_tokens)}
    
    next_idx = len(vocab)
    for token, freq in counter.most_common():
        if token not in vocab:
            if freq >= min_freq:
                vocab[token] = next_idx
                next_idx += 1
                if max_size and len(vocab) >= max_size:
                    break
    
    return vocab

# 构建词汇表
en_vocab = build_vocab(train_data, token_field='en_tokens', min_freq=1)
de_vocab = build_vocab(train_data, token_field='de_tokens', min_freq=1)

# 获取词汇表大小
input_dim = len(de_vocab)   # 源语言（德文）
output_dim = len(en_vocab)  # 目标语言（英文）

权重初始化

In [13]:
# 先创建模型
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

# 然后定义并应用权重初始化
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(36, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(37, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=37, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [14]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(36, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(37, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=37, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [16]:
import torch
import torch.nn as nn
from collections import Counter

# 1. 定义 Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

# 2. 定义 Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

# 3. 定义 Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        _, hidden, cell = self.encoder(src)
        
        input = trg[:, 0]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
            
        return outputs

# 4. 构建词汇表（假设 train_data 已经存在）
def build_vocab(data, token_field, min_freq=1, max_size=None, special_tokens=None):
    if special_tokens is None:
        special_tokens = ['<pad>', '<unk>', '<sos>', '<eos>']
    
    counter = Counter()
    for example in data:
        tokens = example[token_field]
        counter.update(tokens)
    
    vocab = {token: idx for idx, token in enumerate(special_tokens)}
    
    next_idx = len(vocab)
    for token, freq in counter.most_common():
        if token not in vocab:
            if freq >= min_freq:
                vocab[token] = next_idx
                next_idx += 1
                if max_size and len(vocab) >= max_size:
                    break
    
    return vocab

# 5. 创建模型
# 确保 train_data 已经存在并且有 'en_tokens' 和 'de_tokens' 字段
en_vocab = build_vocab(train_data, token_field='en_tokens', min_freq=1)
de_vocab = build_vocab(train_data, token_field='de_tokens', min_freq=1)

input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Input dim (German vocab): {input_dim}")
print(f"Output dim (English vocab): {output_dim}")
print(f"Device: {device}")

# 创建模型
encoder = Encoder(input_dim, encoder_embedding_dim, hidden_dim, n_layers, encoder_dropout)
decoder = Decoder(output_dim, decoder_embedding_dim, hidden_dim, n_layers, decoder_dropout)
model = Seq2Seq(encoder, decoder, device).to(device)

# 6. 初始化权重
def init_weights(m):
    for name, param in m.named_parameters():
        if param.requires_grad:
            nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

# 7. 统计参数
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

print("✓ Model created successfully!")

Input dim (German vocab): 36
Output dim (English vocab): 37
Device: cuda
The model has 7,394,085 trainable parameters
✓ Model created successfully!


In [22]:
# 先定义模型（假设你已经有了 en_vocab 和 de_vocab）
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(input_dim, encoder_embedding_dim, hidden_dim, n_layers, encoder_dropout)
decoder = Decoder(output_dim, decoder_embedding_dim, hidden_dim, n_layers, decoder_dropout)
seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)

# 然后统计参数
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(seq2seq_model):,} trainable parameters")

The model has 7,394,085 trainable parameters


In [23]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 如果你的模型变量名是 seq2seq_model，使用它
print(f"The model has {count_parameters(seq2seq_model):,} trainable parameters")

The model has 7,394,085 trainable parameters


优化器 optimizer

In [24]:
import torch.optim as optim

# 然后创建优化器
optimizer = optim.Adam(seq2seq_model.parameters())
optimizer = optim.Adam(seq2seq_model.parameters())

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


损失函数 Loss Function

In [25]:
# 方法1：直接定义
pad_index = 0
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

# 方法2：从词汇表中获取（推荐）
# 假设你已经有了 en_vocab 或 de_vocab
pad_index = en_vocab.get('<pad>', 0)  # 或者 de_vocab.get('<pad>', 0)
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

# 方法3：如果已经定义了特殊token的索引
PAD_IDX = 0  # 或其他你定义的索引
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [26]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [66]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [67]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

# 模型训练

In [29]:
import spacy
from torchtext.vocab import build_vocab_from_iterator

# 1. 加载模型
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

# 2. Tokenization
def tokenize_example(example):
    return {
        'en_tokens': [token.text for token in en_nlp.tokenizer(example['en'])],
        'de_tokens': [token.text for token in de_nlp.tokenizer(example['de'])]
    }

train_data = train_data.map(tokenize_example)
valid_data = valid_data.map(tokenize_example)
test_data = test_data.map(tokenize_example)

# 3. 构建词汇表
special_tokens = ['<unk>', '<pad>', '<sos>', '<eos>']

def yield_tokens(data, token_field):
    for example in data:
        yield example[token_field]

en_vocab = build_vocab_from_iterator(
    yield_tokens(train_data, 'en_tokens'),
    min_freq=2,
    specials=special_tokens
)

de_vocab = build_vocab_from_iterator(
    yield_tokens(train_data, 'de_tokens'),
    min_freq=2,
    specials=special_tokens
)

en_vocab.set_default_index(en_vocab['<unk>'])
de_vocab.set_default_index(de_vocab['<unk>'])

print(f"✓ 英文词汇表: {len(en_vocab)}")
print(f"✓ 德文词汇表: {len(de_vocab)}")

Map: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 4921.78 examples/s]


✓ 英文词汇表: 6191
✓ 德文词汇表: 8014


In [31]:
import datasets
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.vocab import build_vocab_from_iterator

# ========== 1. 加载数据 ==========
print("加载数据...")
dataset = datasets.load_dataset("bentrevett/multi30k")

train_data = dataset["train"]
valid_data = dataset["validation"]
test_data = dataset["test"]

print(f"训练集大小: {len(train_data)}")
print(f"验证集大小: {len(valid_data)}")
print(f"测试集大小: {len(test_data)}")
print(f"列名: {train_data.column_names}")

# ========== 2. 构建词汇表 ==========
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
sos_token = "<sos>"
eos_token = "<eos>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

print("\n构建英文词汇表...")
en_vocab = build_vocab_from_iterator(
    train_data["en"],
    min_freq=min_freq,
    specials=special_tokens,
)

print("构建德文词汇表...")
de_vocab = build_vocab_from_iterator(
    train_data["de"],
    min_freq=min_freq,
    specials=special_tokens,
)

# 设置默认索引
en_vocab.set_default_index(en_vocab[unk_token])
de_vocab.set_default_index(de_vocab[unk_token])

print(f"\n英文词汇表大小: {len(en_vocab)}")
print(f"德文词汇表大小: {len(de_vocab)}")

# 获取特殊token的索引
PAD_IDX = en_vocab[pad_token]
SOS_IDX = en_vocab[sos_token]
EOS_IDX = en_vocab[eos_token]
UNK_IDX = en_vocab[unk_token]

print(f"PAD_IDX: {PAD_IDX}, SOS_IDX: {SOS_IDX}, EOS_IDX: {EOS_IDX}")

# ========== 3. 设置超参数 ==========
INPUT_DIM = len(de_vocab)   # 源语言（德文）
OUTPUT_DIM = len(en_vocab)  # 目标语言（英文）
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

print(f"\nINPUT_DIM: {INPUT_DIM}")
print(f"OUTPUT_DIM: {OUTPUT_DIM}")
print(f"ENC_EMB_DIM: {ENC_EMB_DIM}")
print(f"HID_DIM: {HID_DIM}")

# ========== 4. 定义模型（如果需要） ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n使用设备: {device}")

# 如果需要查看示例数据
print(f"\n示例数据:")
print(f"英文: {train_data[0]['en']}")
print(f"德文: {train_data[0]['de']}")

加载数据...
训练集大小: 29000
验证集大小: 1014
测试集大小: 1000
列名: ['en', 'de']

构建英文词汇表...
构建德文词汇表...

英文词汇表大小: 82
德文词汇表大小: 99
PAD_IDX: 1, SOS_IDX: 2, EOS_IDX: 3

INPUT_DIM: 99
OUTPUT_DIM: 82
ENC_EMB_DIM: 256
HID_DIM: 512

使用设备: cuda

示例数据:
英文: Two young, White males are outside near many bushes.
德文: Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


In [32]:
from torchtext.vocab import build_vocab_from_iterator

# 定义特殊 token
special_tokens = ['<unk>', '<pad>', '<sos>', '<eos>']

# 构建英文词汇表
def yield_en_tokens(data):
    for example in data:
        yield example['en']

en_vocab = build_vocab_from_iterator(
    yield_en_tokens(train_data),
    min_freq=1,
    specials=special_tokens
)

# 构建德文词汇表
def yield_de_tokens(data):
    for example in data:
        yield example['de']

de_vocab = build_vocab_from_iterator(
    yield_de_tokens(train_data),
    min_freq=1,
    specials=special_tokens
)

# 设置默认索引
en_vocab.set_default_index(en_vocab['<unk>'])
de_vocab.set_default_index(de_vocab['<unk>'])

# 获取特殊 token 的索引
PAD_IDX = en_vocab['<pad>']
SOS_IDX = en_vocab['<sos>']
EOS_IDX = en_vocab['<eos>']
UNK_IDX = en_vocab['<unk>']

print(f"英文词汇表大小: {len(en_vocab)}")
print(f"德文词汇表大小: {len(de_vocab)}")
print(f"PAD_IDX: {PAD_IDX}, SOS_IDX: {SOS_IDX}, EOS_IDX: {EOS_IDX}")

# 设置超参数
INPUT_DIM = len(de_vocab)
OUTPUT_DIM = len(en_vocab)

英文词汇表大小: 84
德文词汇表大小: 103
PAD_IDX: 1, SOS_IDX: 2, EOS_IDX: 3


In [33]:
from torchtext.vocab import build_vocab_from_iterator

# ========== 先构建词汇表 ==========
special_tokens = ['<unk>', '<pad>', '<sos>', '<eos>']

# 构建英文词汇表
def yield_en_tokens(data):
    for example in data:
        yield example['en']

# 构建德文词汇表
def yield_de_tokens(data):
    for example in data:
        yield example['de']

# 构建词汇表
en_vocab = build_vocab_from_iterator(
    yield_en_tokens(train_data),
    min_freq=2,
    specials=special_tokens
)

de_vocab = build_vocab_from_iterator(
    yield_de_tokens(train_data),
    min_freq=2,
    specials=special_tokens
)

# 设置默认索引
en_vocab.set_default_index(en_vocab['<unk>'])
de_vocab.set_default_index(de_vocab['<unk>'])

# 获取特殊token的索引
PAD_IDX = en_vocab['<pad>']
SOS_IDX = en_vocab['<sos>']
EOS_IDX = en_vocab['<eos>']

print(f"英文词汇表大小: {len(en_vocab)}")
print(f"德文词汇表大小: {len(de_vocab)}")

# ========== 然后设置超参数 ==========
INPUT_DIM = len(de_vocab)          # 德语词汇表大小
OUTPUT_DIM = len(en_vocab)         # 英语词汇表大小
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

print(f"INPUT_DIM: {INPUT_DIM}, OUTPUT_DIM: {OUTPUT_DIM}")

英文词汇表大小: 82
德文词汇表大小: 99
INPUT_DIM: 99, OUTPUT_DIM: 82


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# ========== 1. 定义模型类（假设已定义）==========
# class Encoder(nn.Module): ...
# class Decoder(nn.Module): ...
# class Seq2Seq(nn.Module): ...

# ========== 2. 设置超参数（取消注释并赋值）==========
INPUT_DIM = len(de_vocab)          # 德语词汇表大小
OUTPUT_DIM = len(en_vocab)         # 英语词汇表大小
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
BATCH_SIZE = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ========== 3. 实例化模型 ==========
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 初始化权重
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.normal_(param.data, mean=0, std=0.01)

model.apply(init_weights)

print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

# ========== 4. 优化器和损失函数 ==========
optimizer = optim.Adam(model.parameters())
PAD_IDX = en_vocab["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ========== 5. 数据加载器 ==========
def collate_fn(batch):
    de_ids = [torch.tensor(item["de_ids"]) for item in batch]
    en_ids = [torch.tensor(item["en_ids"]) for item in batch]

    de_ids = nn.utils.rnn.pad_sequence(de_ids, batch_first=False, padding_value=de_vocab["<pad>"])
    en_ids = nn.utils.rnn.pad_sequence(en_ids, batch_first=False, padding_value=en_vocab["<pad>"])

    return {"de_ids": de_ids, "en_ids": en_ids}

train_data_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_data_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)

The model has 7,444,818 trainable parameters


In [28]:
import torch
import torch.nn as nn

# ========== 1. Encoder ==========
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src len, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src len, batch size, emb dim]

        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src len, batch size, hid dim * n directions]
        # hidden = [n layers * n directions, batch size, hid dim]
        # cell = [n layers * n directions, batch size, hid dim]

        return hidden, cell

# ========== 2. Decoder ==========
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers, batch size, hid dim]
        # cell = [n layers, batch size, hid dim]

        input = input.unsqueeze(0)  # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, emb dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [1, batch size, hid dim]

        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]

        return prediction, hidden, cell

# ========== 3. Seq2Seq ==========
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src = [src len, batch size]
        # trg = [trg len, batch size]

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        # 存储所有输出
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        # Encoder 最后的状态
        hidden, cell = self.encoder(src)

        # 第一个输入是 <sos>
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            # 决定是否使用 teacher forcing
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[t] if teacher_force else top1

        return outputs

In [35]:
# ========== 先定义所有超参数 ==========
INPUT_DIM = len(de_vocab)          # 德语词汇表大小
OUTPUT_DIM = len(en_vocab)         # 英语词汇表大小
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"INPUT_DIM: {INPUT_DIM}")
print(f"OUTPUT_DIM: {OUTPUT_DIM}")
print(f"Device: {device}")

# ========== 然后创建模型 ==========
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

print("✓ Model created successfully!")

INPUT_DIM: 99
OUTPUT_DIM: 82
Device: cuda
✓ Model created successfully!


In [40]:
# 将 tokens 转换为索引
def numericalize_example(example):
    # 假设你的数据列名是 'en' 和 'de'（tokens列表）
    # 将 tokens 转换为对应的索引
    en_indices = [en_vocab[token] for token in example['en']]
    de_indices = [de_vocab[token] for token in example['de']]
    
    return {
        'en_ids': en_indices,
        'de_ids': de_indices
    }

# 应用到数据集
train_data = train_data.map(numericalize_example)
valid_data = valid_data.map(numericalize_example)
test_data = test_data.map(numericalize_example)

# 验证
print(train_data[0].keys())
print(f"de_ids: {train_data[0]['de_ids'][:10]}")
print(f"en_ids: {train_data[0]['en_ids'][:10]}")

Map: 100%|████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 3613.59 examples/s]

dict_keys(['en', 'de', 'en_ids', 'de_ids'])
de_ids: [41, 27, 5, 7, 4, 50, 12, 6, 18, 5]
en_ids: [29, 17, 9, 4, 26, 9, 21, 7, 15, 30]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import tqdm
from tqdm import tqdm

# ========== 定义训练函数 ==========
def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    model.train()
    epoch_loss = 0
    
    for batch in tqdm(data_loader, desc="Training", leave=False):
        # 获取数据
        src = batch["de_ids"].to(device)  # 源语言（德文）
        trg = batch["en_ids"].to(device)  # 目标语言（英文）
        
        # 清零梯度
        optimizer.zero_grad()
        
        # 前向传播
        output = model(src, trg, teacher_forcing_ratio)
        
        # 计算损失（忽略 <pad>）
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        
        # 反向传播
        loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        # 更新参数
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(data_loader)

# ========== 定义评估函数 ==========
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            
            # 评估时不使用 teacher forcing
            output = model(src, trg, teacher_forcing_ratio=0)
            
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    
    return epoch_loss / len(data_loader)

# ========== 1. 设置超参数 ==========
INPUT_DIM = len(de_vocab)
OUTPUT_DIM = len(en_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
BATCH_SIZE = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"设备: {device}")
print(f"德文词汇表大小: {INPUT_DIM}")
print(f"英文词汇表大小: {OUTPUT_DIM}")

# ========== 2. 实例化模型 ==========
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 初始化权重
def init_weights(m):
    for name, param in m.named_parameters():
        if param.requires_grad:
            nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

# 统计参数
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数量: {count_parameters(model):,}")

# ========== 3. 定义优化器和损失函数 ==========
optimizer = optim.Adam(model.parameters())
PAD_IDX = en_vocab["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ========== 4. 创建数据加载器 ==========
def collate_fn(batch):
    de_ids = [torch.tensor(item["de_ids"]) for item in batch]
    en_ids = [torch.tensor(item["en_ids"]) for item in batch]
    
    de_ids = nn.utils.rnn.pad_sequence(de_ids, batch_first=True, padding_value=de_vocab["<pad>"])
    en_ids = nn.utils.rnn.pad_sequence(en_ids, batch_first=True, padding_value=en_vocab["<pad>"])
    
    return {"de_ids": de_ids, "en_ids": en_ids}

train_data_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_data_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)

print(f"训练批次数量: {len(train_data_loader)}")
print(f"验证批次数量: {len(valid_data_loader)}")

# ========== 5. 开始训练 ==========
n_epochs = 10
clip = 1
teacher_forcing_ratio = 0.5
best_valid_loss = float("inf")

print("\n开始训练...")
for epoch in tqdm(range(n_epochs), desc="Epochs"):
    train_loss = train_fn(model, train_data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device)
    valid_loss = evaluate_fn(model, valid_data_loader, criterion, device)
    
    print(f'\nEpoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')
    
    # 保存最佳模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-model.pt')
        print(f'✓ 保存最佳模型 (Val Loss: {valid_loss:.3f})')

print("\n训练完成！")

In [39]:
import torch
import tqdm

def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    """训练一个 epoch"""
    model.train()
    epoch_loss = 0

    for batch in tqdm.tqdm(data_loader, desc="Training", leave=False):
        src = batch["de_ids"].to(device)  # 德语输入
        trg = batch["en_ids"].to(device)  # 英语目标

        optimizer.zero_grad()

        # 前向传播
        output = model(src, trg, teacher_forcing_ratio)

        # 计算损失（忽略 <pad>）
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)  # 去掉 <sos>
        trg = trg[1:].view(-1)  # 去掉 <sos>

        loss = criterion(output, trg)
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

In [ ]:
# ========== 先创建模型 ==========
# 确保所有必要的变量都已定义
INPUT_DIM = len(de_vocab)
OUTPUT_DIM = len(en_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 创建编码器、解码器和模型
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 初始化权重
def init_weights(m):
    for name, param in m.named_parameters():
        if param.requires_grad:
            nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

# 打印模型参数
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

# ========== 设置优化器和损失函数 ==========
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ========== 训练参数 ==========
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5
best_valid_loss = float("inf")

# ========== 开始训练 ==========
for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    
    print(f'\nEpoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\tVal Loss: {valid_loss:.3f}')
    
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'\tSaved best model!')

In [ ]:
import tqdm  # 添加这行
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

# 模型验证

In [ ]:
# 1. 先创建模型（与保存时的架构相同）
INPUT_DIM = len(de_vocab)
OUTPUT_DIM = len(en_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 创建模型实例
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 2. 然后加载保存的权重
model.load_state_dict(torch.load("tut1-model.pt", map_location=device))

# 3. 设置为评估模式
model.eval()

print("✓ Model loaded successfully!")

In [ ]:
model.load_state_dict(torch.load("tut1-model.pt"))

In [33]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [ ]:
import datasets

# 加载数据集
dataset = datasets.load_dataset("bentrevett/multi30k")

# 定义 train_data, valid_data, test_data
train_data = dataset["train"]
valid_data = dataset["validation"]
test_data = dataset["test"]

# 现在可以使用了
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

print(f"德文: {sentence}")
print(f"英文: {expected_translation}")

In [ ]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

In [ ]:
def translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lower, sos_token, eos_token, device, max_length=50):
    """
    翻译函数
    """
    model.eval()
    
    # 1. Tokenization
    if lower:
        sentence = sentence.lower()
    
    tokens = [token.text for token in de_nlp.tokenizer(sentence)]
    
    # 2. 添加特殊token并转换为索引
    tokens = [sos_token] + tokens + [eos_token]
    
    # 3. 转换为张量
    indices = [de_vocab[token] for token in tokens]
    src_tensor = torch.tensor(indices).unsqueeze(0).to(device)  # (1, seq_len)
    
    # 4. 编码
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # 5. 解码
        trg_indices = [sos_token]
        trg_tensor = torch.tensor([en_vocab[sos_token]]).to(device)
        
        for _ in range(max_length):
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
            pred_token = output.argmax(1).item()
            
            # 转换为token
            for token, idx in en_vocab.get_stoi().items():
                if idx == pred_token:
                    trg_indices.append(token)
                    break
            
            # 如果遇到结束符，停止
            if token == eos_token:
                break
                
            trg_tensor = torch.tensor([pred_token]).to(device)
        
        # 6. 移除特殊token并返回
        translation = ' '.join(trg_indices[1:-1])  # 移除 <sos> 和 <eos>
        
    return translation

# 现在可以调用了
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

print(f"原句: {sentence}")
print(f"翻译: {translation}")
print(f"期望: {expected_translation}")

In [ ]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [ ]:
# 先定义 translate_sentence 函数
def translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, device, max_len=50):
    """
    翻译句子
    """
    model.eval()
    
    # 分词
    tokens = [token.text for token in de_nlp.tokenizer(sentence.lower())]
    tokens = ['<sos>'] + tokens + ['<eos>']
    
    # 转换为索引
    src_indices = [de_vocab[token] for token in tokens if token in de_vocab.get_stoi()]
    src_tensor = torch.tensor(src_indices).unsqueeze(0).to(device)
    
    # 编码
    with torch.no_grad():
        _, hidden, cell = model.encoder(src_tensor)
        
        # 解码
        trg_index = en_vocab['<sos>']
        trg_tensor = torch.tensor([trg_index]).to(device)
        translated_tokens = []
        
        for _ in range(max_len):
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
            pred_index = output.argmax(1).item()
            
            # 获取对应的token
            for token, idx in en_vocab.get_stoi().items():
                if idx == pred_index:
                    if token == '<eos>':
                        return ' '.join(translated_tokens)
                    translated_tokens.append(token)
                    break
            
            trg_tensor = torch.tensor([pred_index]).to(device)
        
        return ' '.join(translated_tokens)

# 选择一个测试句子
test_sentence = test_data[0]["de"]
expected = test_data[0]["en"]

# 进行翻译
translation = translate_sentence(
    test_sentence, 
    model, 
    en_nlp, 
    de_nlp, 
    en_vocab, 
    de_vocab, 
    device
)

# 现在可以打印了
print(f"源语言 (德文): {test_sentence}")
print(f"翻译结果 (英文): {translation}")
print(f"期望输出: {expected}")

# 然后你可以单独使用 translation 变量
print(translation)

In [ ]:
translation